In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

lit = pd.read_csv(r"C:\Users\Harshu\Downloads\Table29.6-States.csv")
enr = pd.read_csv(r"C:\Users\Harshu\Downloads\UDISE_2021_22_Table_5.9_0.csv")
ptr = pd.read_csv(r"C:\Users\Harshu\Downloads\ptr_data.csv")

print("Literacy shape :", lit.shape)
print("Enrollment shape :", enr.shape)
print("PTR shape :", ptr.shape)

print("\nLiteracy Data:")
print(lit.head(3))

print("\nEnrollment Data:")
print(enr.head(3))

print("\nPTR Data:")
print(ptr.head(3))

Literacy shape : (36, 13)
Enrollment shape : (37, 25)
PTR shape : (37, 8)

Literacy Data:
  All India/State/Union Territory  1991 - Male  1991 - Female  1991 - Persons  \
0                       All India           52             64              39   
1                  Andhra Pradesh           44             55              33   
2               Arunachal Pradesh           42             52              30   

   2001 - Male  2001 - Female  2001 - Persons  2011 - Rural - Male  \
0           65             75              54                   77   
1           61             70              50                   69   
2           54             64              44                   67   

   2011 - Rural - Female  2011 - Rural - Person  2011 - Urban - Male  \
0                     58                     68                   89   
1                     52                     60                   86   
2                     52                     60                   88   

   2011 - Urban

In [3]:

df_lit = lit[[
    "All India/State/Union Territory",
    "2011 - Rural - Person",
    "2011 - Urban - Persons"
]].copy()

df_lit.columns = ["State", "Rural", "Urban"]

df_lit["Literacy"] = ((df_lit["Rural"] + df_lit["Urban"]) / 2).round(2)


df_lit = df_lit[df_lit["State"] != "All India"].reset_index(drop=True)

df_enr = enr[[
    "India/State/ UT",
    "Enrolment - Private Unaided Recognized - Total (Pre-Primary to Higher Secondary) - Total"
]].copy()

df_enr.columns = ["State", "Enrollment"]

df_enr["Enrollment"] = pd.to_numeric(df_enr["Enrollment"], errors="coerce")

df_enr = df_enr.dropna().reset_index(drop=True)


df_ptr = ptr[[
    "State Name",
    "PTR (Equal to 4/7)"
]].copy()

df_ptr.columns = ["State", "PTR"]

df_ptr["PTR"] = pd.to_numeric(df_ptr["PTR"], errors="coerce")

df_ptr = df_ptr[df_ptr["State"] != "All India"]

df_ptr = df_ptr.dropna().reset_index(drop=True)


df_final = df_lit.merge(df_enr, on="State").merge(df_ptr, on="State")


df_final["Students_per_Teacher"] = (
    df_final["Enrollment"] / df_final["PTR"]
).round(2)


df_final["Enrollment_Lakh"] = (df_final["Enrollment"] / 100000).round(2)

print("Final dataset shape :", df_final.shape)
print(df_final.head())

Final dataset shape : (30, 8)
               State  Rural  Urban  Literacy  Enrollment    PTR  \
0     Andhra Pradesh     60     80      70.0     3321792  19.45   
1  Arunachal Pradesh     60     83      71.5       94940  42.86   
2              Assam     69     89      79.0     1595135  25.63   
3              Bihar     60     77      68.5     3306257  55.89   
4                Goa     87     90      88.5       39113  17.50   

   Students_per_Teacher  Enrollment_Lakh  
0             170786.22            33.22  
1               2215.12             0.95  
2              62237.03            15.95  
3              59156.50            33.06  
4               2235.03             0.39  


In [4]:

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import panel as pn

pn.extension('plotly', sizing_mode='stretch_width')

print("🎉 Panel + Plotly Ready!")

🎉 Panel + Plotly Ready!


In [5]:
df_lit = lit.rename(columns={'All India/State/Union Territory': 'State'})

chart1 = pn.Column(
    pn.pane.Markdown('### 📊 Rural vs Urban Literacy Comparison'),
    pn.pane.Plotly(
        px.bar(
            df_lit.sort_values('2011 - Rural - Person'),
            x='State',
            y=[
                '2011 - Rural - Person',
                '2011 - Urban - Persons'
            ],
            barmode='group',
            template='plotly'
        ).update_layout(
            xaxis_tickangle=-45,
            height=500,
            width=950
        )
    )
)

chart1

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [6]:

df_lit = lit.rename(columns={'All India/State/Union Territory': 'State'})

df_lit = df_lit[df_lit['State'] != 'All India']

df_lit = df_lit.dropna(subset=['2011 - Urban - Male', '2011 - Urban - Female'])

df_lit = df_lit.sort_values('2011 - Urban - Male')

chart2 = pn.Column(
    pn.pane.Markdown('### 📚 Male vs Female Literacy Gap (Urban 2011)'),

    pn.pane.Plotly(
        px.line(
            df_lit,
            x=['2011 - Urban - Female', '2011 - Urban - Male'],
            y='State',
            orientation='h',
            markers=True,
            template='plotly',
            color_discrete_sequence=['#E91E63', '#1565C0']
        ).update_layout(
            height=700,   
            width=950,
            plot_bgcolor='white'
        )
    )
)

chart2

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [7]:
df_enr = enr.rename(columns={'India/State/ UT': 'State'})

# remove India row
df_enr = df_enr[df_enr['State'] != 'India']

# define columns
cols = [
    'Enrolment - Private Unaided Recognized - Pre-Primary - Total',
    'Enrolment - Private Unaided Recognized - Primary (1 to 5) - Total',
    'Enrolment - Private Unaided Recognized - Upper Primary (6-8) - Total',
    'Enrolment - Private Unaided Recognized - Secondary (9-10) - Total',
    'Enrolment - Private Unaided Recognized - Higher Secondary (11-12) - Total'
]

df_enr[cols] = df_enr[cols].apply(pd.to_numeric, errors='coerce')


df_enr['Total'] = df_enr[cols].sum(axis=1)


df_enr = df_enr[df_enr['Total'] > 50000]


chart3 = pn.Column(
    pn.pane.Markdown('''
## 📊 Education Level-wise Enrollment
> Clear comparison across states
'''),

    pn.pane.Plotly(
        px.bar(
            df_enr,
            x='State',
            y=cols,
            barmode='stack',
            color_discrete_sequence=[
                '#FFB74D', '#4DB6AC', '#9575CD', '#64B5F6', '#F06292'
            ]
        ).update_layout(
            xaxis_tickangle=-65,
            height=700,
            width=1600,
            yaxis_type="log",   
            bargap=0.08,
            font=dict(size=13),

            legend=dict(
                orientation='h',
                y=-0.35,
                x=0.5,
                xanchor='center',
                font=dict(size=10)
            ),

            margin=dict(b=200),
            plot_bgcolor='white',
            paper_bgcolor='white'
        )
    )
)

chart3

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [8]:


chart4 = pn.Column(
    pn.pane.Markdown('## 🎓 Enrollment Share by Top States'),

    pn.pane.Plotly(
        px.pie(
            df_final.sort_values('Enrollment', ascending=False).head(8),
            names='State',
            values='Enrollment',
            hole=0.55,


            color_discrete_sequence=[
                '#1B5E20',  
                '#2E7D32',
                '#388E3C',
                '#43A047',
                '#26A69A',
                '#1E88E5',
                '#3949AB',
                '#6A1B9A'   
            ]
        ).update_traces(
            textinfo='percent+label',
            textfont_size=12,
            pull=[0.06, 0, 0, 0, 0, 0, 0, 0],
            marker=dict(line=dict(width=0))  
        ).update_layout(
            height=520,
            width=900,

            legend=dict(
                font=dict(size=11),
                x=1.05,
                y=0.9
            ),

            plot_bgcolor='white',
            paper_bgcolor='white'
        )
    )
)

chart4

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [9]:
chart5 = pn.Column(
    pn.pane.Markdown('## 📊 Education Distribution & Insights'),

    pn.pane.Plotly(
        px.scatter(
            df_final,
            x='Literacy',
            y='PTR',
            size='Enrollment',
            color='Enrollment',

            color_continuous_scale='Viridis',

            hover_name='State',
            size_max=40
        ).update_traces(
            marker=dict(
                opacity=0.75,
                line=dict(width=0)   
            )
        ).update_layout(
            height=550,
            width=900,
            title='Literacy vs PTR (Bubble Size = Enrollment)',

          
           
            xaxis=dict(showgrid=True, gridcolor='#E0E0E0'),
            yaxis=dict(showgrid=True, gridcolor='#E0E0E0'),


            coloraxis_colorbar=dict(
                title='Enrollment',
                thickness=15
            ),

            font=dict(size=13)
        )
    )
)

chart5

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [10]:
chart7 = pn.Column(
    pn.pane.Markdown('### 🔥 Education Metrics Correlation Heatmap'),

    pn.pane.Plotly(
        px.imshow(
            df_final[
                ['Rural', 'Urban', 'Literacy', 'Enrollment', 'PTR', 'Students_per_Teacher']
            ].corr(),

            color_continuous_scale='RdBu_r',   
            zmin=-1, zmax=1,

            text_auto='.2f'
        ).update_layout(
            height=700,
            width=950,
            title='Correlation Between Education Metrics',


            plot_bgcolor='white',
            paper_bgcolor='white',

  
            coloraxis_colorbar=dict(
                title='Correlation',
                thickness=15
            ),

            font=dict(size=13)
        )
    )
)

chart7

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [11]:
chart12 = pn.Column(
    pn.pane.Markdown('### 📈 Literacy Trend Across States'),

    pn.pane.Plotly(
        px.line(
            df_final.sort_values('Literacy'),
            x='State',
            y='Literacy',
            markers=True,

            line_shape='spline'
        ).update_traces(
            line=dict(color='#1E88E5', width=3), 
            marker=dict(size=8, color='#1565C0')
        ).update_layout(
            xaxis_tickangle=-60,
            height=550,
            width=950,

            title='State-wise Literacy Trend',


            plot_bgcolor='white',
            paper_bgcolor='white',


            xaxis=dict(showgrid=False),
            yaxis=dict(showgrid=True, gridcolor='#E0E0E0'),

            font=dict(size=13)
        )
    )
)

chart12

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [12]:
chart13 = pn.Column(
    pn.pane.Markdown('### 📊 State-wise Enrollment (Lakh)'),

    pn.pane.Plotly(
        px.bar(
            df_final.sort_values('Enrollment_Lakh'),
            x='State',
            y='Enrollment_Lakh',
            color='Enrollment_Lakh',

            color_continuous_scale='Viridis'
        ).update_layout(
            xaxis_tickangle=-60,
            height=550,
            width=950,

            title='Enrollment Distribution Across States',

            plot_bgcolor='white',
            paper_bgcolor='white',

            yaxis=dict(showgrid=True, gridcolor='#E0E0E0'),
            xaxis=dict(showgrid=False),

            font=dict(size=13)
        )
    )
)

chart13

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [13]:
chart14 = pn.Column(
    pn.pane.Markdown('### 🥧 Enrollment Share by Top States'),

    pn.pane.Plotly(
        px.pie(
            df_final.sort_values('Enrollment', ascending=False).head(10),
            names='State',
            values='Enrollment',

            color_discrete_sequence=px.colors.sequential.Viridis
        ).update_traces(
            textinfo='percent+label',
            textfont_size=12,

            marker=dict(line=dict(width=0)),

            pull=[0.06] + [0]*9
        ).update_layout(
            height=520,
            width=900,

            title='Enrollment Distribution (Top 10 States)',

            plot_bgcolor='white',
            paper_bgcolor='white',

            legend=dict(
                font=dict(size=11),
                x=1.05,
                y=0.9
            ),

            margin=dict(t=60, b=40, l=40, r=120)
        )
    )
)

chart14

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [14]:

df_temp = df_final.sort_values('Students_per_Teacher')

chart6 = pn.Column(
    pn.pane.Markdown('### 👨‍🏫 Students per Teacher Analysis'),

    pn.pane.Plotly(
        go.Figure()

        .add_bar(
            x=df_temp['State'],
            y=df_temp['Students_per_Teacher'],
            marker=dict(
                color=df_temp['Students_per_Teacher'],
                colorscale=[
                    [0, '#E0F2F1'],
                    [0.5, '#26A69A'],
                    [1, '#004D40']
                ]
            ),
            name='Students per Teacher'
        )

        .add_scatter(
            x=df_temp['State'],
            y=df_temp['Students_per_Teacher'],
            mode='lines+markers',
            line=dict(color='#1E88E5', width=3),
            marker=dict(size=6),
            name='Trend'
        )

        .update_layout(
            height=600,
            width=1200,
            title='Students per Teacher Ratio by State',

            xaxis=dict(
                tickangle=-60,
                showgrid=False
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor='#E0E0E0'
            ),

            plot_bgcolor='white',
            paper_bgcolor='white',

            font=dict(size=13),
            legend=dict(x=1.02)
        )
    )
)

chart6



chart6.servable() 
chart6

Column(sizing_mode='stretch_width')
    [0] Markdown(str, sizing_mode='stretch_width')
    [1] Plotly(Figure, sizing_mode='stretch_width')

In [15]:

df_final['State'] = df_final['State'].str.strip()

state_options = sorted(df_final['State'].dropna().unique())

state_filter = pn.widgets.MultiChoice(
    name='🎯 Select States',
    options=['All States'] + state_options,
    value=['All States'],
    height=250
)

def filter_data(states):
    if not states or 'All States' in states:
        return df_final
    return df_final[df_final['State'].isin(states)]

In [17]:

df_final['State'] = df_final['State'].str.strip()

state_options = sorted(df_final['State'].dropna().unique())

state_filter = pn.widgets.MultiChoice(
    name='🎯 Select States',
    options=['All States'] + state_options,
    value=['All States'],
    height=150   
)

def filter_data(states):
    if not states or 'All States' in states:
        return df_final
    return df_final[df_final['State'].isin(states)]



template = pn.template.FastListTemplate(
    title='🎓 Indian Education Dashboard ',
    header_background='#2C3E50',


    sidebar=[


        pn.pane.Markdown('''
# 🎓 Education Dashboard
### India's Learning Insights
''', styles={
            'background': '#2C3E50',
            'color': 'white',
            'padding': '10px',   
            'border-radius': '10px',
            'margin-bottom': '8px'
        }),


        pn.Column(
            pn.pane.Markdown("### 🎯 Filter by State"),
            state_filter,
            styles={
                'background': '#F5F7FA',
                'padding': '8px',   
                'border-radius': '10px',
                'box-shadow': '0 2px 6px rgba(0,0,0,0.08)',
                'margin-bottom': '8px'
            }
        ),

   
        pn.pane.Markdown('''
### 📌 About Project
- 📚 Literacy Rates  
- 🎓 Student Enrollment  
- 👨‍🏫 Student-Teacher Ratio  
''', styles={
            'background': '#E3F2FD',
            'padding': '8px',
            'border-radius': '10px',
            'border-left': '5px solid #1E88E5',
            'margin-bottom': '8px'
        }),


        pn.pane.Markdown('''
### 🔍 Coverage
- 📊 State comparison  
- 📈 Trends  
- 📉 PTR vs Literacy  
''', styles={
            'background': '#E8F5E9',
            'padding': '8px',
            'border-radius': '10px',
            'border-left': '5px solid #43A047',
            'margin-bottom': '8px'
        }),


        pn.pane.Markdown('''
### 💡 Insights
- Higher literacy → better education  
- High PTR → lower attention  
''', styles={
            'background': '#FFF3E0',
            'padding': '8px',
            'border-radius': '10px',
            'border-left': '5px solid #FB8C00',
            'margin-bottom': '8px'
        }),

        pn.pane.Markdown('''
### 📁 Data Sources  
data.gov.in  
UDISE+  
''', styles={
            'background': '#F3E5F5',
            'padding': '8px',
            'border-radius': '10px',
            'border-left': '5px solid #8E24AA',
            'margin-bottom': '8px'
        }),
    ],

    main=[

        pn.Row(

            pn.Column(
                pn.bind(lambda states:
                    pn.indicators.Number(
                        name='Total States',
                        value=len(filter_data(states)),
                        font_size='32pt'
                    ), state_filter),
                styles={
                    'background': '#E3F2FD',
                    'border-left': '6px solid #1E88E5',
                    'padding': '12px',
                    'border-radius': '10px',
                    'box-shadow': '0 3px 8px rgba(0,0,0,0.1)'
                }
            ),

            pn.Column(
                pn.bind(lambda states:
                    pn.indicators.Number(
                        name='Avg Literacy',
                        value=round(filter_data(states)['Literacy'].mean(),1),
                        format='{value}%',
                        font_size='32pt'
                    ), state_filter),
                styles={
                    'background': '#E8F5E9',
                    'border-left': '6px solid #43A047',
                    'padding': '12px',
                    'border-radius': '10px',
                    'box-shadow': '0 3px 8px rgba(0,0,0,0.1)'
                }
            ),

            pn.Column(
                pn.bind(lambda states:
                    pn.indicators.Number(
                        name='Total Enrollment',
                        value=int(filter_data(states)['Enrollment'].sum()/1e6),
                        format='{value}M',
                        font_size='32pt'
                    ), state_filter),
                styles={
                    'background': '#FFF3E0',
                    'border-left': '6px solid #FB8C00',
                    'padding': '12px',
                    'border-radius': '10px',
                    'box-shadow': '0 3px 8px rgba(0,0,0,0.1)'
                }
            ),

            pn.Column(
                pn.bind(lambda states:
                    pn.indicators.Number(
                        name='Avg PTR',
                        value=round(filter_data(states)['PTR'].mean(),1),
                        font_size='32pt'
                    ), state_filter),
                styles={
                    'background': '#F3E5F5',
                    'border-left': '6px solid #8E24AA',
                    'padding': '12px',
                    'border-radius': '10px',
                    'box-shadow': '0 3px 8px rgba(0,0,0,0.1)'
                }
            ),
        ),


        pn.Column(chart1),
        pn.Column(chart2),
        pn.Column(chart3),
        pn.Column(chart4),
        pn.Column(chart5),
        pn.Column(chart6),
        pn.Column(chart12),
        pn.Column(chart13),
        pn.Column(chart14),
        pn.Column(chart7),
    ],

    accent_base_color='#2C3E50'
)

template.show()

Launching server at http://localhost:50486
